In [1]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import functions as func
from utils import parameters as params
from utils import metabolites as metab

from tqdm import tqdm
import copy
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/other/test_lp/'

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


In [2]:
jabba = Tru
counter = 1
base = 0
mu_val = 1e-9
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'

In [ ]:
# from expression import build_me_model
# tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
#                                             dummy_protein = False)

# if jabba:
#     for r in tme.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
    
    
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [3]:
with open(lp_path + 'working_version_' + str(base) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme1 = pickle.load(handle)
    
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)    
sln1, stat1, _ = tme1.solve_lp(mu_val = mu_val)    

In [5]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme0.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln0[tme0.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.899296e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,3.280437e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11


In [6]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme1.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln1[tme1.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.126854e-10
unmodeled_protein_biomass_to_biomass,unmodeled_protein_biomass_to_biomass,2.054807e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,1.997807e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11


In [ ]:
# with open(lp_path + 'working_version' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [ ]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

S = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(tme.reactions)]})
res.index = [r.id for r in tme.reactions]

if stat == 0:
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)
    print('Model did not solve')
    
res.loc[[i for i in res.index if 'biomass' in i],:].sort_values(by = 'reaction_fluxes', ascending = False)

In [ ]:
def save_me_model(me_model, counter):
    print('Success, please update git')
#     lp_path = '/data2/hratch/human_me/test_lp/'
#     me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [4]:
# S_0 = pd.read_hdf(fn, key = str(base))
# S_1 = pd.read_hdf(fn, key = str(counter))

In [5]:
# S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    save_me_model(tme, counter)

Columns are not the same
Rows are not the same
Dataframes are not equal due to column/row label mismatch, will not save model


In [ ]:
# idx = S_1.index.tolist()
# for i in range(len(idx)):
#     val = idx[i]
#     if 'folded' in val and 'polyub' in val:
#         idx[i] = val.replace('protein_polyub', 'protein_' + val[-1] + '_polyub')
# S_1.index = idx

# idx = set(S_0.index).intersection(S_1.index)
# col = set(S_0.columns).intersection(S_1.columns)
# S_0 = S_0.loc[idx, col]
# S_1 = S_1.loc[idx, col]

print(set(S_1.index).difference(S_0.index))
print(set(S_0.index).difference(S_1.index))

for tr, v in dict(zip(list(set(S_1.index).difference(S_0.index)), list(set(S_0.index).difference(S_1.index)))).items():
    S_1.index = pd.Series(S_1.index).replace(to_replace = tr, value = v, inplace = False)

In [ ]:
mismatch, mm, mm_2 = get_changes(S_1, S_0)

In [ ]:
list(mm.keys())[:10]

In [ ]:
m_idx = 124
mm_2[m_idx]

In [ ]:
mm[m_idx]

In [ ]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

In [ ]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

In [ ]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-6]

In [ ]:
test

In [ ]:
# res0 = pd.read_csv(lp_path + 'works_trash.csv', index_col = 0)
# res['og_fluxes'] = res0.loc[res.index.tolist(), :]['reaction_fluxes'].tolist()
# res['diff'] = res['reaction_fluxes'] - res['og_fluxes']

# testing ubiquitin cleavage

In [ ]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    
